In [2]:
import pickle
from pathlib import Path

# Question_Generation/
PROJECT_ROOT = Path.cwd().parent

EMBEDDINGS_FILE = (
    PROJECT_ROOT
    / "data"
    / "embeddings"
    / "question_embeddings.pkl"
)

print("Loading:", EMBEDDINGS_FILE)

with open(EMBEDDINGS_FILE, "rb") as f:
    embedded_questions = pickle.load(f)

print("Loaded successfully!")
print("Number of records:", len(embedded_questions))

Loading: C:\Users\User\RAG SYS\Question_Generation\data\embeddings\question_embeddings.pkl
Loaded successfully!
Number of records: 2150


In [3]:
first = embedded_questions[0]

print("ID:", first["id"])
print("Embedding shape:", first["embedding"].shape)
print("Question:", first["metadata"]["question"])

ID: ai_ml_computer_vision_q001
Embedding shape: (384,)
Question: What is Computer Vision and what problems does it solve?


In [4]:
import numpy as np

vectors = np.array(
    [item["embedding"] for item in embedded_questions],
    dtype="float32"
)

print("Vectors shape:", vectors.shape)
print("Data type:", vectors.dtype)

Vectors shape: (2150, 384)
Data type: float32


In [5]:
# ============================================================
# BUILD FAISS VECTOR INDEX
# ============================================================

import faiss

# Number of dimensions in each embedding
dimension = vectors.shape[1]

print("Embedding dimension:", dimension)

# Inner Product index
# Because our embeddings were normalized,
# Inner Product = Cosine Similarity
index = faiss.IndexFlatIP(dimension)

# Add all vectors
index.add(vectors)

print("=" * 70)
print("FAISS INDEX CREATED")
print("=" * 70)

print("\nNumber of vectors in index:", index.ntotal)
print("Vector dimension:", index.d)

Embedding dimension: 384
FAISS INDEX CREATED

Number of vectors in index: 2150
Vector dimension: 384


In [6]:
# ============================================================
# SAVE FAISS INDEX
# ============================================================

from pathlib import Path

# Question_Generation/
PROJECT_ROOT = Path.cwd().parent

# Vector store directory
VECTOR_STORE_DIR = PROJECT_ROOT / "vector_store"
VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

# FAISS index file
INDEX_FILE = VECTOR_STORE_DIR / "question_index.faiss"

# Save index
faiss.write_index(index, str(INDEX_FILE))

print("=" * 70)
print("FAISS INDEX SAVED")
print("=" * 70)

print("\nFile:")
print(INDEX_FILE)

print("\nVectors saved:", index.ntotal)
print("\n✓ Save complete")

FAISS INDEX SAVED

File:
C:\Users\User\RAG SYS\Question_Generation\vector_store\question_index.faiss

Vectors saved: 2150

✓ Save complete


In [7]:
# ============================================================
# SAVE METADATA MAPPING
# ============================================================

import pickle

METADATA_FILE = VECTOR_STORE_DIR / "metadata.pkl"

# Keep the same order as the vectors in FAISS
metadata = [
    {
        "id": item["id"],
        "metadata": item["metadata"]
    }
    for item in embedded_questions
]

with open(METADATA_FILE, "wb") as f:
    pickle.dump(metadata, f)

print("=" * 70)
print("METADATA SAVED")
print("=" * 70)

print("\nFile:")
print(METADATA_FILE)

print("\nRecords saved:", len(metadata))

print("\nFirst metadata:")
print(metadata[0])

print("\n✓ Save complete")

METADATA SAVED

File:
C:\Users\User\RAG SYS\Question_Generation\vector_store\metadata.pkl

Records saved: 2150

First metadata:
{'id': 'ai_ml_computer_vision_q001', 'metadata': {'id': 'ai_ml_computer_vision_q001', 'original_id': 'Q001', 'title': 'What is Computer Vision?', 'question': 'What is Computer Vision and what problems does it solve?', 'type': 'technical', 'track': 'ai_ml', 'category': 'computer_vision', 'topic': 'computer_vision_basics', 'difficulty': 'easy', 'experience': ['intern', 'junior'], 'skills': ['computer_vision'], 'language': 'en', 'duration': 60, 'source': 'question_bank', 'expected_concepts': ['Image understanding', 'Video understanding', 'Computer vision systems', 'Image analysis', 'Object recognition'], 'source_file': 'ai_ml\\computer_vision.md'}}

✓ Save complete


In [9]:
# ============================================================
# RETRIEVAL QUALITY TEST
# ============================================================

test_queries = [
    "Explain object detection in computer vision.",
    "What is Docker and why is it used?",
    "How does Kubernetes manage containers?",
    "Explain Python inheritance.",
    "What is a neural network?",
    "How do SQL databases handle relationships?",
    "What is overfitting in machine learning?",
    "Explain REST APIs.",
]

TOP_K = 5

for query in test_queries:

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    scores, indices = index.search(query_embedding, TOP_K)

    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]), start=1
    ):

        result = metadata[idx]

        print(
            f"{rank}. "
            f"[{score:.4f}] "
            f"{result['metadata']['question']}"
        )


QUERY: Explain object detection in computer vision.
1. [0.7911] What is object detection?
2. [0.7547] What is the difference between image classification and object detection?
3. [0.7394] What is image segmentation?
4. [0.7385] What is face detection?
5. [0.7384] What is a bounding box in object detection?

QUERY: What is Docker and why is it used?
1. [0.7639] Why should you use a `.dockerignore` file?
2. [0.7604] What is Docker and why is it commonly used in DevOps?
3. [0.7532] Why are Docker volumes used?
4. [0.7517] What is Docker and what problem does it solve?
5. [0.7506] Why are Docker volumes used?

QUERY: How does Kubernetes manage containers?
1. [0.7333] What is container orchestration and why is it needed?
2. [0.7320] What is the difference between a Pod and a container?
3. [0.7251] Design a production Kubernetes platform for a system containing a frontend, backend API, database, background workers, and machine learning model serving.
4. [0.7245] What is Kubernetes and what 